In [1]:
import matplotlib.pyplot as plt
import parselib
import sys
import numpy as np
import matplotlib as mpl
import plotconfig

In [2]:
args = {
    "chaos": "on_1",
    "timestamp": ["20251119", "20251119", "2025112", "2025112"],
    "cca": ["cubic", "bbr", "bbr2", "bbr3"],
    "test_cca": "yes",
    "kernel": ["kernel6-1", "zkernel5-13-BBRv2", "zkernel6-13-BBRv3"],
    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [100],
    "delay_rtt": [10],
    "deadline_run": [1000000, 10000000],
}
metric = "bits_per_second"

baselogpath = "../data"


In [3]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 2160
end 2160
bytes 2160
bits_per_second 2160
mbps_timeseries 2160
rttms_timeseries 2160
retransmits 2160
timestamp 2160
iteration 2160
cpu_host_total 2160
cpu_host_user 2160
cpu_host_system 2160
cpu_remote_total 2160
chaos 2160
deadline_run 2160
deadline_period 2160
os 2160
bdp 2160
setup 2160
cca 2160
cpus 2160
kernel 2160
mode 2160
loss 2160
rate 2160
delay_rtt 2160
buffer_size_bytes 2160
parallel 2160
socket_buffer 2160
app_buffer 2160
n 2160
sysctl_cmd 2160
vm 2160
bandwidth_delay_product 2160
loss_mode 2160
vms 2160
pacing 2160
hyperthreading 2160
tso 2160
qdisc 2160
hpet 2160
tsc 2160
hostq 2160
loadperc 2160
deadline_period_factor 2160
random_loss_rate 2160
gemodel_q 2160
original_cca 2160
test_cca 2160
default_qdisc 2160
json 2160


In [4]:
df["slice_perc"] = (df["deadline_run"]/df["deadline_period"])*100
df["mbps"] = df["bits_per_second"]/1000000

replace_label = {
    "bbr": "BBRv1",
    "bbr2": "BBRv2",
    "bbr3": "BBRv3",
    "cubic": "Cubic"
}

In [5]:
def timeseries(df, cca, slice=25, savefig=False):  
    if savefig:
        mpl.use('agg')
    plotconfig.configure_conext()
    width = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)
    height = width/2*3/4
    FIG_SIZE = (width, height)
          
    fig, axes = plt.subplots(3,2, figsize=FIG_SIZE, sharey=True,constrained_layout=True)

    for en_tsl, abs_timeslice in enumerate(df["deadline_run"].unique()):
        data_ = df[df["deadline_run"] == abs_timeslice]
    
        for it, cca in enumerate(["bbr", "bbr2", "bbr3"]):
            data = data_[data_["cca"] == cca]
            data = data[round(data["slice_perc"]) == slice]

            if cca == "bbr" or cca == "cubic":
                data = data[data["kernel"] == "kernel6-1"]

            first_idx = data.index.values[0]
            bli_cnt =0
            for idx, row in data.iterrows():
                if row['mbps_timeseries'][0] < 20:
                    bli_cnt += 1
                axes[it][en_tsl].plot(row['mbps_timeseries'], color=plotconfig.COLORS[0], marker="|", alpha=0.4, linewidth=1, label=replace_label[cca] if idx == first_idx else "")
            
            data_cubic = df[df["cca"] == "cubic"]
            data_cubic = data_cubic[data_cubic["slice_perc"] == slice]
            data_cubic = data_cubic[data_cubic["kernel"] == "kernel6-1"]
            first_idx = data_cubic.index.values[0]
            for idx, row in data_cubic.iterrows():
                axes[it][en_tsl].plot(row['mbps_timeseries'], color=plotconfig.COLORS[3], marker="|", alpha=0.4, linewidth=1, label=replace_label["cubic"] if idx == first_idx else "")
            
            axes[it][en_tsl].set_xticks([])


        axes[it][en_tsl].set_ylim(0, int(args["rate"][0])+40)
        axes[it][en_tsl].set_xlim(0,19)
        axes[it][en_tsl].tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-2)

    axes[1][0].set_ylabel("Throughput [mbps]",fontsize=plotconfig.FONT_SIZE-2)
    axes[2][0].set_xlabel("Run time [sec]", fontsize=plotconfig.FONT_SIZE-2)
    axes[2][1].set_xlabel("Run time [sec]", fontsize=plotconfig.FONT_SIZE-2)
    axes[2][0].set_xticks([1,3,5,7,9,11,13,15,17,19], [2,4,6,8,10,12,14,16,18,20],fontsize=plotconfig.FONT_SIZE-2)
    axes[2][1].set_xticks([1,3,5,7,9,11,13,15,17,19], [2,4,6,8,10,12,14,16,18,20],fontsize=plotconfig.FONT_SIZE-2)
    axes[0][1].legend(fontsize=plotconfig.FONT_SIZE-2,loc="lower right", framealpha=0.9, bbox_to_anchor=(1.25,0))
    axes[1][1].legend(fontsize=plotconfig.FONT_SIZE-2,loc="lower right", framealpha=0.9, bbox_to_anchor=(1.25,0))
    axes[2][1].legend(fontsize=plotconfig.FONT_SIZE-2,loc="lower right", framealpha=0.9, bbox_to_anchor=(1.25,0))

    if savefig:
        fig.savefig(f"figures/figure_5.pdf", format="pdf")
    else:
        plt.show()

In [6]:
timeseries(df, "bbr", 25, True)
